## Counterfactual explanations: Практика


Среди библиотек, позволяющих реализовать Counterfactual explanations наиболее популярной является [DiCE](https://github.com/interpretml/DiCE/tree/main). Именно с DiCE (Diverse Counterfactual Explanations), мы будем работать, так как она наиболее гибка для использования. Объяснения можно получать для моделей, обученных при помощи `sklearn, keras, tensorflow` и `pytorch`, но только для табличных данных.

Чуть менее известной библиотекой является [CARLA](https://github.com/carla-recourse/CARLA). Если вы решаете задачу классификации, рекомендуем к ней присмотреться, так как она более широка *по способам* получения контрфактического объяснения.


In [ ]:
!pip install -q dice-ml "pandas>2"


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

data = pd.read_csv('https://github.com/SadSabrina/open-xai-materials/raw/main/data/fetch_california_housing.csv',
                   index_col=0)
data.head()


**Контрфактическое объяснение: зачем?**

Контрфактическое объяснение используется в локальном смысле — то есть, когда объясняется прогноз модели для какого-то конкретного объекта из тестовых данных.

**Кейс:** рассмотрим задачу прогнозирования цен дома. Вы предоставили модель непосредственному заказчику и необходимо узнать, почему конкретный дом $x[i]$ был оценен в стоимость $y_i$? Какие минимальные изменения необходимо внести, чтобы прогноз принадлежал отрезку $[y_{j-1}, y_j]$?


**Что дает контрфактическое объяснение в случае с регрессией?**

**Ответ:** Как один из возможных вариантов, наблюдение $x$, такое что $\hat{f}(x)$ принадлежит заданному диапазону $[y_i, y_j]$.

Чтобы не отходить далеко от концепции, разобранной в примере, продолжим работать с нашей прекрасной Калифорнией и ее не менее чудесными домами. Заметим, что наш датасет содержит только непрерывные признаки. Для нас это хорошо, поскольку мы можем не углубляться в предобработку данных. Чтобы совсем избежать шага предобработки, будем использовать ансамблевый алгоритм `RandomForestRegressor`.

**Q1.** Какие еще методы объяснения мы могли бы применить к алгоритму `Random Forest` из `sklearn`?

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/1) · [шаг на Степике](https://stepik.org/lesson/1263846/step/2)

In [ ]:
#Подготовим данные для обучения

X = data.drop('target', axis=1)
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from IPython.display import display

forest = RandomForestRegressor(random_state=42)
params = {'max_depth': [6, 10, 14]}

gs = GridSearchCV(forest, params, cv=3, scoring='neg_mean_squared_error')
gs.fit(X_train, y_train)

estimator = gs.best_estimator_

print('Cv score:')
display(pd.DataFrame(gs.cv_results_))
print('Test MSE:', mean_squared_error(estimator.predict(X_test), y_test))


##**DiCE**


Чтобы найти ближайший контрфактический объект, мы должны определить:
- по какому критерию искать (контрфактность прогноза)
- где искать
- как искать
- для кого искать

Отвечать на эти вопросы, используя DiCE (как для классификации, так и для регрессии), мы будем определять объекты двух классов: Data и Model. В случае, если вы хотите сохранить приватность данных, то DiCE также предоставляет эту возможность (ее мы также разберем чуть ниже).

Для использования `Data` и `Model` необходимо:

- сохранить названия непрерывных и категориальных признаков в списки `continuous_features` и `discrete_features` соответственно
- обучить модель

Свой запрос к контрфактическому объяснению мы формулируем как:

`Во всем наборе данных (его мы тебе дадим) найди ближайшие контрфакты для выбранного экземпляра, такие, что прогноз $\hat{f}(x)$ для него $\in [y_i, y_j]$`

1. **Где искать** — указываем в объекте класса `Data`. Это может быть любое множество данных, структура которого такая же, как и у данных, на которых обучена модель.
2. **Как искать** — здесь DiCE предлагает три метода поиска `kdtree`, `genetic` и `random`. От выбранного метода в том числе зависят найденные контрфакты. То есть такого, что вы нашли 3 контрфакта и они устойчивы от метода поиска к методу поиска **не будет**. Устойчивые от итерации к итерации контрфакты дает `kdtree`, поэтому мы рекомендуем использовать его.
3. **Для кого искать** - любой объект из тестового набора данных.


**Где искать**

Искать контрфактический пример можно как в тренировочном, так и в тестовом наборах данных.

**Тренировочный набор данных**

Цель: отладка модели в процессе обучения.

**Преимущества:**
- Можно использовать все доступные данные для анализа, так как обучающая выборка — самое большое доступное нам множество данных.
- Можно лучше понять, как модель обучается и какие признаки наиболее влияют на предсказания. Это в некоторых случаях помогает убрать нежелательные объекты или скорректировать распределения некоторых признаков.

**Недостатки:**
- Контрфактические примеры могут не отражать реальность, если модель переобучена на тренировочных данных.
- Найденные примеры могут быть не релевантными для новых, ранее невидимых данных.

**Тестовый набор данных**

Цель: Оценка и интерпретация модели на новых, невидимых данных.

**Преимущества:**
- Позволяет проверить, как модель будет вести себя в реальных сценариях.

**Недостатки:**
- Меньшее количество данных для анализа по сравнению с тренировочным набором.

**Общие рекомендации:**

- Для исследовательских целей и интерпретации модели во время ее разработки используйте тренировочный набор данных. Это поможет лучше понять внутреннюю работу модели и ее реакции на изменения признаков.
- Для оценки модели и ее поведения на новых данных: используйте тестовый набор данных. Это поможет проверить, как модель будет работать в реальных условиях и насколько устойчивы ее предсказания к изменениям в данных.


Мы рассмотрим сценарий поиска контрфактического объекта в тестовых данных `X_test`. Хотя для определения класса `Data` необходимо, чтобы используемый набор полностью удовлетворял структуре данных, на которых обучена модель, нам не обязательно включать в `test` значения целевой переменной. Достаточно поставить на них "заглушку" любым значением.


In [ ]:
sample = X_test.sample(n=20, random_state=42)
data_to_search = sample.copy()

X_train_dice = X_train.copy()
X_train_dice['target'] = y_train


In [ ]:
import dice_ml
from dice_ml import Dice

continuous_features= list(X.columns)

d_housing = dice_ml.Data(dataframe=X_train_dice, outcome_name='target', continuous_features=continuous_features)

m_housing = dice_ml.Model(model=estimator, backend="sklearn", model_type='regressor')


Реализация поиска контрфактического объяснения проводится при помощи объекта класса `Dice`. На выбор в DiCE включены *3 метода поиска*: `genetic`, `kdtree`, `random`.

**На что влияет метод поиска:**
- на **скорость** поиска — (I) `random`, (II) `genetic`, (III) `kdtree`, время меняется в зависимости от количества наблюдений, для которых ищутся контрфактические наблюдения (не обязательно искать только для одного экземпляра данных)
- на **устойчивость** поиска — найденный контрфактический объект не обязан быть единственным или лучшим из-за чего методы поиска могут выдавать разные объекты от итерации к итерации. Наиболее устойчив (проверено на малых данных) `kdtree`. Мы рекомендуем использовать его.


In [ ]:
exp_genetic_housing = Dice(d_housing, m_housing, method="kdtree") #инициализируем объект, при помощи которого будем искать контрфактическое объяснение

query_instance_housing = data_to_search[:1]

print('Model prediction:', """Ваш код здесь""")


In [ ]:
query_instance_housing


**Зададимся вопросом:** как минимально поменять объект, чтобы прогноз модели увеличился в 2 раза?

**Q2.** Задайте отрезок для поиска под данную задачу вида: $[y_{twice},\ y_{twice}+0.5]$.

В ответ запишите правую границу (округлив до сотых). Введите число в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/2) · [шаг на Степике](https://stepik.org/lesson/1263846/step/3)

In [ ]:
genetic_housing_twice = exp_genetic_housing.generate_counterfactuals(query_instance_housing,
                                                               total_CFs=3,
                                                               desired_range=...)  # ваш код здесь: [y_twice, y_twice + 0.5]
genetic_housing_twice.visualize_as_dataframe(show_only_changes=True)


**Q3.** Задайте отрезок для поиска: $[4, 5]$

В ответ введите значение переменной `Population` для любого из найденных объектов. Введите число в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/3) · [шаг на Степике](https://stepik.org/lesson/1263846/step/4)

In [ ]:
genetic_housing_twice_up = exp_genetic_housing.generate_counterfactuals(query_instance_housing,
                                                               total_CFs=3,
                                                               desired_range=...)  # ваш код здесь: [4, 5]
genetic_housing_twice_up.visualize_as_dataframe(show_only_changes=False)


**Какой из найденных ближайший? Евклидова метрика**

Расстояние между объектами $x, y$ по определению — это совсем не $\sqrt{\sum_i^n (x_i - y_i)^2}$.

В математике расстояние является *метрикой* и объявляется аксиоматически. Не пугайтесь, вам не обязательно знать аксиомы. Из этого факта вам необходимо понимать, что расстояние — это один из способов оценить удаленность объектов друг от друга.

Расстояние, которое вводят в школе ($p(x, y) = \sqrt{\sum_i^n (x_i - y_i)^2}$) — это лишь один из примеров. Но им можно и нужно пользоваться. Оно называется Евклидовым расстоянием. Если ваши табличные данные отражают специфическую структуру, например, текст (допустим, в виде Tf-idf-кодирования), то может быть продуктивнее рассмотреть другой вид расстояния.  


In [ ]:
# Сохраним найденные объекты в виде массивов: для 2x интервала и для отрезка [4, 5]
cf_objects = genetic_housing_twice.cf_examples_list[0].final_cfs_df.values
cf_objects_4_5 = ...  # ваш код здесь (объекты, найденные для отрезка [4, 5])


Вычислим Евклидово расстояние между объясняемым и найденными примерами. Расстояние будем считать по всем координатам, кроме целевой.


In [ ]:
def euclidean_distance(x, y):

  distance = np.sqrt(np.sum((x-y)**2))
  return distance


def cheb_distance(x, y):

  distance = np.max(np.abs((x-y)))
  return distance


In [ ]:
query_instance_housing_as_vector = query_instance_housing.values


**Q4.** Посчитайте Евклидово расстояние между найденными контрфактическими объектами для отрезка `[4, 5]` и объясняемым примером. **Чему равно наименьшее Евклидово расстояние?**

Ответ округлите до сотых. Введите число в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/4) · [шаг на Степике](https://stepik.org/lesson/1263846/step/5)

In [ ]:
# последняя координата каждого найденного объекта - target, в расстояние она не входит
for i in cf_objects_4_5:
  print('Euclidean dist:', ...)  # ваш код здесь


Если же нам просто важно, чтобы изменения были наименьшими в каждой координате, то можно вычислить расстояние Чебышева по формуле $p(x, y) = max(|x_i-y_i|)$

**Q5.** Рассчитайте расстояние Чебышева для тех же объектов. Поменялся ли результат? Выберите ответ, который соответствует выводам из получившихся значений.

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/5) · [шаг на Степике](https://stepik.org/lesson/1263846/step/6)

In [ ]:
for i in cf_objects_4_5:
  print('Chebyshev dist:', ...)  # ваш код здесь


**Q6.** Проанализируйте сами объект номер 7, не меняя способ поиска объяснений. Контрфактические объекты ищите в отрезке $[model\_prediction\cdot 2, model\_prediction\cdot 2 + 0.5]$.

Чему равно наибольшее значение целевой переменной среди найденных 3х объектов? Ответ округлите до сотых.

Объект номер 7 - строка `data_to_search.iloc[7:8]` (нумерация с нуля, как у объекта `data_to_search[:1]` выше). Таргет найденных объектов берите из столбца `target` таблицы, которую печатает `visualize_as_dataframe`.

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/6) · [шаг на Степике](https://stepik.org/lesson/1263846/step/7)

In [ ]:
query_instance_housing_two = ...  # ваш код здесь: объект номер 7
print('Model prediction', estimator.predict(query_instance_housing_two))

genetic_housing_seven = exp_genetic_housing.generate_counterfactuals(query_instance_housing_two,
                                                               total_CFs=3,
                                                               desired_range=...)  # ваш код здесь

genetic_housing_seven.visualize_as_dataframe(show_only_changes=True)


**Работа в DiCE без доступа к тренировочным данным**

На самом деле, мы с вами уже почти разобрали этот шаг. DiCE является методом, который больше зависит от модели и того, как она была обучена, соответственно показывать тренировочные данные нет необходимости. Но некоторые статистики данных знать все-таки нужно, поскольку методу нужно понимать, где искать. Иначе результатов метода не получить.

Итак, чтобы работать с анонимизированными данными достаточно перечислить все признаки и их возможные значения в атрибуте `features`. Соберем простой пример — будем прогнозировать класс уже знакомых вам Ирисов, чтобы не переобучать новую модель долго.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True, as_frame=True)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=42)

model = DecisionTreeClassifier(max_depth=3)
model.fit(X_tr, y_tr)

print('Model accuracy:', accuracy_score(model.predict(X_te), y_te))


Рассмотрим список признаков и диапазоны их значений.


In [ ]:
X_tr.describe()


Собираем данные, указывая границы числовых признаков. Для категориальных необходимо было бы перечислить категории. Порядок признаков должен быть тот же, что и при обучении модели.


In [ ]:

d_without_access = dice_ml.Data(features={'sepal length (cm)': [4.3, 7.7],
                                          'sepal width (cm)': [2, 4.2],
                                          'petal length (cm)': [1.1, 6.7],
                                          'petal width (cm)' : [0.1, 2.5]},
                 outcome_name='target')


Класс model объявляем практически так же. Вместо модели, обученной непосредственно в течение ноутбука, можно также добавить путь к сохраненной предобученной модели в параметр 'model_path'.


In [ ]:
backend = 'sklearn'
m = dice_ml.Model(model=model, backend=backend, model_type='classifier')

query_instance = pd.DataFrame({'sepal length (cm)': 12,
                               'sepal width (cm)': 3,
                               'petal length (cm)': 1,
                               'petal width (cm)': 0.7},
                               index=[0])


Объявляем "поиск" и получаем результаты. Однако история без доступа к данным имеет значительный минус — невозможно искать наиболее устойчивым kdtree.


In [ ]:
exp = dice_ml.Dice(d_without_access, m, method="genetic")

dice_exp = exp.generate_counterfactuals(query_instance, total_CFs=3, desired_class=2,  initialization="random"
                                       )

dice_exp.visualize_as_dataframe(show_only_changes=True)


**Q7.** Получилось ли получить устойчивые контрфактические объяснения без доступа к сэмплу данных?

Чтобы проверить, запустите поиск еще раз (ячейка ниже) и сравните найденные объекты с предыдущим запуском. Поиск `genetic` случаен, и примерно в одном-двух запусках из ста два набора могут случайно совпасть; если совпали, запустите ячейку еще раз.

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/7) · [шаг на Степике](https://stepik.org/lesson/1263846/step/8)

In [ ]:
dice_exp_again = exp.generate_counterfactuals(query_instance, total_CFs=3, desired_class=2, initialization="random")

dice_exp_again.visualize_as_dataframe(show_only_changes=True)


**Q8.** Выберите все верные утверждения про метод DiCE.

Выберите ответ в тренажере.

Тот же поиск можно запустить для модели другого устройства: DiCE обращается к модели только за прогнозами. Ниже вместо дерева решений взята логистическая регрессия, а данные и запрос те же.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/9/8) · [шаг на Степике](https://stepik.org/lesson/1263846/step/9)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_tr, y_tr)
print('Model accuracy:', accuracy_score(log_reg.predict(X_te), y_te))

m_log_reg = dice_ml.Model(model=log_reg, backend='sklearn', model_type='classifier')
exp_log_reg = dice_ml.Dice(d_without_access, m_log_reg, method="genetic")

dice_exp_log_reg = exp_log_reg.generate_counterfactuals(query_instance, total_CFs=3, desired_class=2, initialization="random")

dice_exp_log_reg.visualize_as_dataframe(show_only_changes=True)


## **Выводы**
В этом разделе мы познакомились с объяснением моделей при помощи генерации контрфактических объяснений при помощи DiCE. Мы выяснили, что:
- необходимо четко формулировать сэмпл данных, на основе которого будут генерироваться контрфактические значения
- метод не всегда приносит устойчивые объяснения
- для уточнения близости могут быть использованы метрики расстояний
- метод не требует прямого доступа к тренировочным данным.
